# 03 - Data Quality and Cleaning Rules

**Goal:** collect evidence on the **development set only** for each anomaly from notebook 01, then apply the
same deterministic cleaning rules (`src/data_processing.py`) to both development and hold-out.

**Principles**
- Rules are functions with **fixed thresholds**; nothing is fitted, so applying them to the hold-out set leaks nothing.
- Each threshold has a stated basis: domain plausibility or a target-free check (not a bad-rate-driven cut).
- **No imputation:** invalid or missing values become `NaN` and get their own bin in the WoE step.
- Rows are dropped only when the record is impossible.

In [1]:
import sys
sys.path.insert(0, "../src")

from pathlib import Path
for _p in ("../data/processed", "../outputs/figures", "../outputs/metrics", "../outputs/scorecard"):
    Path(_p).mkdir(parents=True, exist_ok=True)   # Git does not keep empty folders

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from data_processing import clean, validate_clean, LATE_COLS, MISSING_GROUPS, MIN_VALID_INCOME, MAX_VALID_UTIL

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

dev = pd.read_csv("../data/processed/raw_dev.csv", index_col="id")
hold = pd.read_csv("../data/processed/raw_holdout.csv", index_col="id")
inc, dr, util = dev["monthly_income"], dev["debt_ratio"], dev["revolving_util"]
print("Evidence below uses the development set only:", dev.shape)

Evidence below uses the development set only: (104987, 11)


## Part A - Evidence (development set)

### A1. Is `debt_ratio` a ratio? Split by income status

In [2]:
status = np.select([inc.isna(), inc < MIN_VALID_INCOME], ["missing", f"< {MIN_VALID_INCOME} (zero/placeholder)"], f">= {MIN_VALID_INCOME}")
dev.groupby(status).apply(lambda d: pd.Series({
    "rows": len(d),
    "share_debt_ratio_gt_1": (d.debt_ratio > 1).mean(),
    "median_debt_ratio": d.debt_ratio.median(),
    "bad_rate": d.target.mean(),
})).round(3)

,rows,share_debt_ratio_gt_1,median_debt_ratio,bad_rate
< 100 (zero/placeholder),1576.0,0.925,705.00,0.035
>= 100,82563.0,0.042,0.29,0.071
missing,20848.0,0.940,1159.00,0.055


**Reading:** when income is valid, `debt_ratio` behaves like a ratio (median ~0.3, few values above 1). When income is
missing or tiny, more than 90% of values exceed 1 and the median is in the hundreds or thousands: it is an
absolute amount, not a ratio, and cannot be compared with the rest (rules R3 and R4).

### A2. Where does income stop being a placeholder? (target-free)

In [3]:
pos = dev[inc > 0]
pos.groupby(pd.cut(pos.monthly_income, [0, 1, 10, 50, 100, 200, 500, 1000, 1e9]), observed=True) \
   .debt_ratio.agg(rows="size", median_debt_ratio="median").round(2)

,rows,median_debt_ratio
monthly_income,,
"(0.0, 1.0]",429,486.50
"(1.0, 10.0]",9,147.67
"(10.0, 50.0]",14,23.48
"(50.0, 100.0]",21,10.96
"(100.0, 200.0]",55,1.77
"(200.0, 500.0]",318,1.26
"(500.0, 1000.0]",1392,0.39
"(1000.0, 1000000000.0]",80785,0.29


**Reading:** the median `debt_ratio` falls from the hundreds (income of 1) to about 11 for income in (50, 100] and to ~1.8 in
(100, 200]. Below ~100 per month the "ratio" is still an absolute amount divided by a placeholder. We therefore set
`MIN_VALID_INCOME = 100` (rule R3). This was chosen from the behaviour of `debt_ratio`, **not** from the bad rate.

### A3. `debt_ratio` tail when income is valid

In [4]:
v = dev[inc >= MIN_VALID_INCOME]
v.groupby(pd.cut(v.debt_ratio, [-1, 0.5, 1, 2, 5, 100, 1e9]), observed=True).target.agg(rows="size", bad_rate="mean").round(3)

,rows,bad_rate
debt_ratio,,
"(-1.0, 0.5]",64418,0.061
"(0.5, 1.0]",14640,0.100
"(1.0, 2.0]",2695,0.136
"(2.0, 5.0]",621,0.098
"(5.0, 100.0]",188,0.074
"(100.0, 1000000000.0]",1,1.000


**Reading:** once zero/placeholder incomes are excluded the extreme tail almost disappears (about 190 rows between 5 and 100, a single row above 100), so no
extra rule is needed. The bad rate still peaks around 1-2 and then declines, a mild non-monotonic pattern. In notebook 04 the business trend (increasing risk) is enforced, so these ~800 rows are absorbed into the top bin (> 0.65) rather than modelled separately.

### A4. `revolving_util` tail

In [5]:
dev.groupby(pd.cut(util, [-1, 0.5, 1, 2, 10, 100, 1e12]), observed=True).target.agg(rows="size", bad_rate="mean").round(3)

,rows,bad_rate
revolving_util,,
"(-1.0, 0.5]",76046,0.027
"(0.5, 1.0]",26619,0.154
"(1.0, 2.0]",2057,0.394
"(2.0, 10.0]",90,0.244
"(10.0, 100.0]",13,0.231
"(100.0, 1000000000000.0]",162,0.037


**Reading:** values above 10 mean a balance above 10x the credit limit, which is not plausible (values reach tens of thousands, i.e.
dollar balances stored in a ratio column). We use `MAX_VALID_UTIL = 10` on **domain** grounds (rule R5). Only a few rows lie
in (10, 100], so the exact cut inside that band has negligible effect; the bad rates above are shown for transparency, not used to place the cut.

### A5. Special codes 96 / 98

In [6]:
special = dev[LATE_COLS].isin([96, 98])
flag = special.any(axis=1)
print("rows with 96/98:", flag.sum(), "| cells:", int(special.sum().sum()), "| rows with all 3 columns:", special.all(axis=1).sum())
print(f"bad rate: {dev.loc[flag, 'target'].mean():.2%} (96/98 rows) vs {dev.loc[~flag, 'target'].mean():.2%} (others)")

rows with 96/98: 170 | cells: 510 | rows with all 3 columns: 170
bad rate: 58.82% (96/98 rows) vs 6.60% (others)


These are special codes with very different risk. The information is kept as `late_special_code_flag`; the counts become `NaN` and get their own bin (rule R2).

### A6. Duplicates

In [7]:
d = dev[dev.duplicated()]
late0 = lambda x: (x[LATE_COLS] == 0).all(axis=1).mean()
pd.DataFrame({
    "duplicate_copies": [len(d), len(dev)],
    "bad_rate": [d.target.mean(), dev.target.mean()],
    "income_missing": [d.monthly_income.isna().mean(), dev.monthly_income.isna().mean()],
    "no_late_history": [late0(d), late0(dev)],
    "util_equals_0": [(d.revolving_util == 0).mean(), (dev.revolving_util == 0).mean()],
}, index=["duplicates", "all development rows"]).T.round(3)

,duplicates,all development rows
duplicate_copies,414.000,104987.000
bad_rate,0.034,0.067
income_missing,0.843,0.199
no_late_history,0.918,0.798
util_equals_0,0.536,0.073


**Reading:** duplicates are **not** random. They concentrate in the income-missing, no-delinquency, low-risk segment (bad rate about half the average).
They are ~0.4% of rows, so we keep them (rule R6) and document the issue. The split already keeps identical rows on the same side.

### A7. Consistency checks

In [8]:
print("n_real_estate > n_open_lines :", (dev.n_real_estate > dev.n_open_lines).sum())
print("revolving_util > 0 with n_open_lines == 0 :", ((util > 0) & (dev.n_open_lines == 0)).sum(), "(kept: lines may have closed)")
print("age >= 100 :", (dev.age >= 100).sum(), "(kept: few rows, falls into the top age bin)")
print("min age in development:", dev.age.min())

n_real_estate > n_open_lines : 0
revolving_util > 0 with n_open_lines == 0 : 1264 (kept: lines may have closed)
age >= 100 : 10 (kept: few rows, falls into the top age bin)
min age in development: 21


### A8. Keep the absolute debt amount as a separate variable? (rejected)

R4 discards `debt_ratio` where it is an absolute amount. We test whether that amount carries risk information before throwing it away.

In [9]:
_tmp, _ = clean(dev)
m = _tmp.monthly_income.isna()
amt = dev.loc[_tmp.index[m], "debt_ratio"]
y = _tmp.loc[m, "target"]
q = pd.qcut(amt.rank(method="first"), 8, labels=False)
print(pd.DataFrame({"amount_min": amt.groupby(q).min().round(0), "amount_max": amt.groupby(q).max().round(0),
                    "rows": y.groupby(q).size(), "bad_rate": y.groupby(q).mean().round(4)}))
rho, p = spearmanr(amt, y)
print(f"Spearman(amount, target) = {rho:.3f} (p = {p:.3f})")

            amount_min  amount_max  rows  bad_rate
debt_ratio                                        
0                  0.0        19.0  2803    0.0485
1                 19.0       125.0  2803    0.0442
2                125.0       575.0  2803    0.0817
3                575.0      1126.0  2803    0.0646
4               1126.0      1681.0  2803    0.0471
5               1681.0      2358.0  2803    0.0428
6               2358.0      3389.0  2803    0.0410
7               3390.0    329664.0  2803    0.0574
Spearman(amount, target) = -0.012 (p = 0.082)


**Reading:** the bad rate is not monotonic in the amount and the rank correlation is about zero, so the amount adds no usable signal. We do not create an extra variable.

## Part B - Apply the rules to both sets

In [10]:
dev_clean, log_dev = clean(dev)
hold_clean, log_hold = clean(hold)
validate_clean(dev_clean)
validate_clean(hold_clean)
print("Development log"); display(log_dev)
print("Hold-out log"); display(log_hold)

Development log


,rule,rows_affected,cells_affected,rows_after
0,R1 drop rows with age < 18,0,0,104987
1,R2 recode 96/98 in late-count columns to NaN +...,170,510,104987
2,R3 monthly_income < 100 (zero / placeholder) -...,1576,1576,104987
3,R4 debt_ratio -> NaN where monthly_income is NaN,22424,22424,104987
4,R5 revolving_util > 10 -> NaN,175,175,104987
5,R6 duplicated rows: kept (no action),0,0,104987


Hold-out log


,rule,rows_affected,cells_affected,rows_after
0,R1 drop rows with age < 18,1,1,45012
1,R2 recode 96/98 in late-count columns to NaN +...,99,297,45012
2,R3 monthly_income < 100 (zero / placeholder) -...,701,701,45012
3,R4 debt_ratio -> NaN where monthly_income is NaN,9584,9584,45012
4,R5 revolving_util > 10 -> NaN,66,66,45012
5,R6 duplicated rows: kept (no action),0,0,45012


In [11]:
for name, raw_, c_ in [("development", dev, dev_clean), ("hold-out", hold, hold_clean)]:
    print(f"{name}: rows {len(raw_):,} -> {len(c_):,} | bad rate {raw_.target.mean():.4%} -> {c_.target.mean():.4%}")
miss = dev_clean.isna().sum().to_frame("n_missing")
miss["pct"] = (miss.n_missing / len(dev_clean) * 100).round(2)
miss[miss.n_missing > 0]

development: rows 104,987 -> 104,987 | bad rate 6.6865% -> 6.6865%
hold-out: rows 45,013 -> 45,012 | bad rate 6.6781% -> 6.6782%


,n_missing,pct
revolving_util,175,0.17
n_30_59_late,170,0.16
debt_ratio,22424,21.36
monthly_income,22424,21.36
n_90_late,170,0.16
n_60_89_late,170,0.16
n_dependents,2723,2.59


Bad rate of the `NaN` groups in the development set (each will become a separate WoE bin):

In [12]:
for col in ["monthly_income", "revolving_util", "n_dependents", "n_90_late"]:
    g = dev_clean[col].isna()
    print(f"{col:15s} NaN: n={g.sum():>6,}  bad rate={dev_clean.loc[g, 'target'].mean():.2%}")

monthly_income  NaN: n=22,424  bad rate=5.34%
revolving_util  NaN: n=   175  bad rate=5.14%
n_dependents    NaN: n= 2,723  bad rate=4.55%
n_90_late       NaN: n=   170  bad rate=58.82%


### Shared missing patterns (plan for notebooks 04-05)

Some columns are `NaN` on exactly the same rows, so their "Missing" WoE bins coincide. With one WoE column per variable this does
**not** create perfect collinearity (the columns still differ on all other rows; notebook 05 checks VIF), but the risk of the shared group is
spread over several coefficients and cannot be attributed to one variable. We verify the shared patterns here and keep them in mind in notebook 05.

In [13]:
for name, cols in MISSING_GROUPS.items():
    same = all((dev_clean[cols[0]].isna() == dev_clean[c].isna()).all() for c in cols[1:])
    print(f"{name:18s} columns {cols} share an identical NaN pattern: {same} ({dev_clean[cols[0]].isna().sum():,} rows)")

income_invalid     columns ['monthly_income', 'debt_ratio'] share an identical NaN pattern: True (22,424 rows)
late_special_code  columns ['n_30_59_late', 'n_90_late', 'n_60_89_late'] share an identical NaN pattern: True (170 rows)


In [14]:
dev_clean.to_csv("../data/processed/dev.csv")
hold_clean.to_csv("../data/processed/holdout.csv")
log_dev.to_csv("../outputs/metrics/cleaning_log_dev.csv", index=False)
log_hold.to_csv("../outputs/metrics/cleaning_log_holdout.csv", index=False)
print("Saved dev.csv, holdout.csv and the cleaning logs")

Saved dev.csv, holdout.csv and the cleaning logs


## Summary of rules

| Rule | Action | Basis |
|---|---|---|
| R1 | Drop age < 18 | Impossible record |
| R2 | 96/98 -> NaN + `late_special_code_flag` | Special codes, very different risk |
| R3 | income < 100 -> NaN | `debt_ratio` is an absolute amount below this level (target-free check) |
| R4 | `debt_ratio` -> NaN where income is NaN | Not a ratio when income is missing/invalid |
| R5 | `revolving_util` > 10 -> NaN | Domain plausibility |
| R6 | Duplicates kept | ~0.4% of rows, documented, split keeps them together |

Next: notebook 04 - binning, WoE and IV on `dev.csv`.